In [3]:
import re
import json
from pathlib import Path

def wildcard_to_regex(pattern):
    """
    Elsevier wildcard → strict regex
    contoh: fertil* → fertil[a-zA-Z]*
    """
    p = re.escape(pattern)
    p = p.replace(r"\*", r"[a-zA-Z]*")
    return r"\b" + p + r"\b"

def match_rule_list(rule_list, text):
    found = 0
    matched = []

    for rule in rule_list:
        for pat in rule["regex"]:
            if re.search(pat, text, flags=re.IGNORECASE):
                found += 1
                matched.append(rule["pattern"])
                break

    return {
        "found": found,
        "matched_keywords": matched
    }

SDG_CORES = {
    "SDG01": ["poverty", "income", "low-income", "poor", "livelihood"],
    "SDG02": ["food", "nutrition", "hunger", "agriculture", "malnutrition"],
    "SDG03": ["health", "disease", "mortality", "diabetes", "epidemi", "healthcare"],
    "SDG04": ["education", "school", "learning", "literacy", "student"],
    "SDG05": ["gender", "women", "female", "equality", "empowerment"],
    "SDG06": ["water", "sanitation", "wastewater", "hygiene"],
    "SDG07": ["energy", "electricity", "renewable", "power"],
    "SDG08": ["employment", "labor", "jobs", "economic", "productivity"],
    "SDG09": ["industry", "infrastructure", "innovation", "manufacturing"],
    "SDG10": ["inequality", "equity", "disparity", "inclusion"],
    "SDG11": ["urban", "city", "housing", "transport"],
    "SDG12": ["consumption", "production", "waste", "recycling"],
    "SDG13": ["climate", "carbon", "emission", "global warming"],
    "SDG14": ["marine", "ocean", "fish", "coast", "coral"],
    "SDG15": ["biodiversity", "forest", "ecosystem", "land"],
    "SDG16": ["governance", "justice", "institution", "policy", "peace"],
    "SDG17": ["partnership", "cooperation", "aid", "development assistance"]
}


def sdg_core_check(sdg, text):
    if sdg not in SDG_CORES:
        return True
    txt = text.lower()
    return any(core in txt for core in SDG_CORES[sdg])

WHITELIST = {"hiv", "gdp", "sme"}

def normalize_rule_list(rule_list):
    out = []
    for x in rule_list:
        if isinstance(x, str):
            out.append({"pattern": x})
        elif isinstance(x, dict) and "pattern" in x:
            out.append(x)
    return out

def filter_short_patterns(rule_json):
    def ok(p):
        base = p["pattern"].replace("*", "")
        return len(base) > 3 or base.lower() in WHITELIST

    for sec in ["include", "exclude"]:
        for cat in rule_json[sec]:
            rule_json[sec][cat] = [
                p for p in normalize_rule_list(rule_json[sec][cat]) if ok(p)
            ]
    return rule_json

def load_rule_json(path):
    with open(path) as f:
        data = json.load(f)

    for sec in ["include", "exclude"]:
        for cat in data[sec]:
            data[sec][cat] = normalize_rule_list(data[sec][cat])
            for item in data[sec][cat]:
                item["regex"] = [wildcard_to_regex(item["pattern"])]

    return filter_short_patterns(data)

def load_all_sdg_rules(folder):
    rules = {}
    for i in range(1, 18):
        sdg = f"SDG{i:02d}"
        r = load_rule_json(Path(folder) / f"{sdg}.json")
        r["sdg_code"] = sdg
        rules[sdg] = r
    return rules

def compute_sdg_score(
    rule,
    title,
    abstract,
    keywords,
    w_title=4,
    w_abstract=1,
    w_keyword=3,
    cap_title=4,
    cap_abstract=8,
    cap_keyword=4,
    soft_exclude_penalty=0.4
):
    # ---- EXCLUDE ----
    ex_t = match_rule_list(rule["exclude"]["TITLE_ABS"], title)
    ex_a = match_rule_list(rule["exclude"]["TITLE_ABS"], abstract)
    ex_k = match_rule_list(rule["exclude"]["AUTHKEY"], keywords)
    excluded = (ex_t["found"] + ex_a["found"] + ex_k["found"]) > 0

    # ---- INCLUDE ----
    in_t = match_rule_list(rule["include"]["TITLE_ABS"], title)
    in_a = match_rule_list(rule["include"]["TITLE_ABS"], abstract)
    in_k = match_rule_list(rule["include"]["AUTHKEY"], keywords)

    t = min(in_t["found"], cap_title)
    a = min(in_a["found"], cap_abstract)
    k = min(in_k["found"], cap_keyword)

    raw = t*w_title + a*w_abstract + k*w_keyword
    denom = cap_title*w_title + cap_abstract*w_abstract + cap_keyword*w_keyword
    score = raw/denom if denom else 0

    # ---- CORE DOMAIN ----
    fulltext = f"{title} {abstract} {keywords}"
    if score > 0 and not sdg_core_check(rule["sdg_code"], fulltext):
        raw = 0
        score = 0

    # ---- SOFT EXCLUDE ----
    if excluded:
        if raw == 0:
            score = 0
        else:
            score *= (1 - soft_exclude_penalty)

    # ---- REASONING ----
    reasons = []
    if in_t["matched_keywords"]:
        reasons.append(f"title: {in_t['matched_keywords']}")
    if in_a["matched_keywords"]:
        reasons.append(f"abstract: {in_a['matched_keywords']}")
    if in_k["matched_keywords"]:
        reasons.append(f"keywords: {in_k['matched_keywords']}")
    if excluded:
        reasons.append("soft-exclude applied")

    return {
        "raw_score": raw,
        "score": score,
        "reasoning": reasons,
        "excluded": excluded
    }

def calibrate_score(score):
    """
    Smooth confidence (logistic-like)
    """
    if score <= 0:
        return 0
    return round(1 - (1 / (1 + 5*score)), 3)

def classify_text_rule_based(
    title,
    abstract,
    keywords,
    rule_folder="/content",
    threshold=0.15
):
    rules = load_all_sdg_rules(rule_folder)
    results = {}

    for sdg, rule in rules.items():
        r = compute_sdg_score(rule, title, abstract, keywords)
        r["confidence"] = calibrate_score(r["score"])
        results[sdg] = r

    labels = [
        sdg for sdg, r in results.items()
        if r["confidence"] >= threshold
    ]

    top = max(results, key=lambda x: results[x]["confidence"])

    return {
        "labels": labels,
        "top_sdg": top,
        "details": results
    }


In [5]:
# SDGS ASLI: SDG 3 | SDG 10

title = "Diabetes mortality and trends before 25 years of age: an analysis of the Global Burden of Disease Study 2019"
abstract = """
Background Diabetes, particularly type 1 diabetes, at younger ages can be a largely preventable cause of death with the correct health care and services. We aimed to evaluate diabetes mortality and trends at ages younger than 25 years globally using data from the Global Burden of Diseases, Injuries, and Risk Factors Study (GBD) 2019.
Methods We used estimates of GBD 2019 to calculate international diabetes mortality at ages younger than 25 years in 1990 and 2019. Data sources for causes of death were obtained from vital registration systems, verbal autopsies, and other surveillance systems for 1990–2019. We estimated death rates for each location using the GBD Cause of Death Ensemble model. We analysed the association of age-standardised death rates per 100 000 population with the Socio-demographic Index (SDI) and a measure of universal health coverage (UHC) and described the variability within SDI quintiles. We present estimates with their 95% uncertainty intervals.
Findings In 2019, 16 300 (95% uncertainty interval 14 200 to 18 900) global deaths due to diabetes (type 1 and 2 combined) occurred in people younger than 25 years and 73·7% (68·3 to 77·4) were classified as due to type 1 diabetes. The age-standardised death rate was 0·50 (0·44 to 0·58) per 100 000 population, and 15 900 (97·5%) of these deaths occurred in low to high-middle SDI countries. The rate was 0·13 (0·12 to 0·14) per 100 000 population in the high SDI quintile, 0·60 (0·51 to 0·70) per 100 000 population in the low-middle SDI quintile, and 0·71 (0·60 to 0·86) per 100 000 population in the low SDI quintile. Within SDI quintiles, we observed large variability in rates across countries, in part explained by the extent of UHC (r2=0·62). From 1990 to 2019, age-standardised death rates decreased globally by 17·0% (−28·4 to −2·9) for all diabetes, and by 21·0% (–33·0 to −5·9) when considering only type 1 diabetes. However, the low SDI quintile had the lowest decline for both all diabetes (−13·6% [–28·4 to 3·4]) and for type 1 diabetes (−13·6% [–29·3 to 8·9]).
Interpretation Decreasing diabetes mortality at ages younger than 25 years remains an important challenge, especially in low and low-middle SDI countries. Inadequate diagnosis and treatment of diabetes is likely to be major contributor to these early deaths, highlighting the urgent need to provide better access to insulin and basic diabetes education and care. This mortality metric, derived from readily available and frequently updated GBD data, can help to monitor preventable diabetes-related deaths over time globally, aligned with the UN's Sustainable Development Targets, and serve as an indicator of the adequacy of basic diabetes care for type 1 and type 2 diabetes across nations.
"""
keywords = ""

result = classify_text_rule_based(title, abstract, keywords)

print("TOP SDG:", result["top_sdg"])
print("DETECTED SDGs:", result["labels"])

print("\n--- REASONING ---")
for sdg, d in result["details"].items():
    if d["confidence"] > 0:
        print(sdg, d["confidence"], "→", d["reasoning"])


TOP SDG: SDG03
DETECTED SDGs: ['SDG03', 'SDG04']

--- REASONING ---
SDG03 0.586 → ["title: ['diabetes', 'disease', 'mortality']", "abstract: ['diabetes', 'diseases', 'health', 'mortality', 'universal health coverage']", 'soft-exclude applied']
SDG04 0.4 → ["abstract: ['education', 'better', 'people', 'access', 'adequacy', 'early', 'education*', 'development', 'develop*', 'Develop*', 'sustainable', 'sustainable development', 'educat*', 'sustainab*', 'sustainable* develop*', 'Sustainable', 'Sustainab*', 'access*']", 'soft-exclude applied']


In [6]:
# SDGS ASLI: SDG 10

title = "Global, regional, and national burden of osteoarthritis,1990–2020 and projections to 2050: a systematic analysisfor the Global Burden of Disease Study 2021"
abstract = """
Background Osteoarthritis is the most common form of arthritis in adults, characterised by chronic pain and loss ofmobility. Osteoarthritis most frequently occurs after age 40 years and prevalence increases steeply with age. WHO hasdesignated 2021–30 the decade of healthy ageing, which highlights the need to address diseases such as osteoarthritis,which strongly affect functional ability and quality of life. Osteoarthritis can coexist with, and negatively effect, otherchronic conditions. Here we estimate the burden of hand, hip, knee, and other sites of osteoarthritis acrossgeographies, age, sex, and time, with forecasts of prevalence to 2050.
Methods In this systematic analysis for the Global Burden of Disease Study, osteoarthritis prevalence in 204 countriesand territories from 1990 to 2020 was estimated using data from population-based surveys from 26 countries for kneeosteoarthritis, 23 countries for hip osteoarthritis, 42 countries for hand osteoarthritis, and US insurance claims for allof the osteoarthritis sites, including the other types of osteoarthritis category. The reference case definition wassymptomatic, radiographically confirmed osteoarthritis. Studies using alternative definitions from the reference casedefinition (for example self-reported osteoarthritis) were adjusted to reference using regression models. Osteoarthritisseverity distribution was obtained from a pooled meta-analysis of sources using the Western Ontario and McMasterUniversities Arthritis Index. Final prevalence estimates were multiplied by disability weights to calculate years livedwith disability (YLDs). Prevalence was forecast to 2050 using a mixed-effects model.
Findings Globally, 595 million (95% uncertainty interval 535–656) people had osteoarthritis in 2020, equal to7·6% (95% UI 6·8–8·4) of the global population, and an increase of 132·2% (130·3–134·1) in total cases since 1990.Compared with 2020, cases of osteoarthritis are projected to increase 74·9% (59·4–89·9) for knee, 48·6% (35·9–67·1)for hand, 78·6% (57·7–105·3) for hip, and 95·1% (68·1–135·0) for other types of osteoarthritis by 2050. The globalage-standardised rate of YLDs for total osteoarthritis was 255·0 YLDs (119·7–557·2) per 100 000 in 2020, a9·5% (8·6–10·1) increase from 1990 (233·0 YLDs per 100 000, 109·3–510·8). For adults aged 70 years and older,osteoarthritis was the seventh ranked cause of YLDs. Age-standardised prevalence in 2020 was more than 5·5% in allworld regions, ranging from 5677·4 (5029·8–6318·1) per 100 000 in southeast Asia to 8632·7 (7852·0–9469·1)per 100 000 in high-income Asia Pacific. Knee was the most common site for osteoarthritis. High BMI contributed to20·4% (95% UI –1·7 to 36·6) of osteoarthritis. Potentially modifiable risk factors for osteoarthritis such as recreationalinjury prevention and occupational hazards have not yet been explored in GBD modelling.
Interpretation Age-standardised YLDs attributable to osteoarthritis are continuing to rise and will lead to substantialincreases in case numbers because of population growth and ageing, and because there is no effective cure forosteoarthritis. The demand on health systems for care of patients with osteoarthritis, including joint replacements,which are highly effective for late stage osteoarthritis in hips and knees, will rise in all regions, but might be out ofreach and lead to further health inequity for individuals and countries unable to afford them. Much more can andshould be done to prevent people getting to that late stage.
"""
keywords = ""

result = classify_text_rule_based(title, abstract, keywords)

print("TOP SDG:", result["top_sdg"])
print("DETECTED SDGs:", result["labels"])

print("\n--- REASONING ---")
for sdg, d in result["details"].items():
    if d["confidence"] > 0:
        print(sdg, d["confidence"], "→", d["reasoning"])


TOP SDG: SDG01
DETECTED SDGs: ['SDG01', 'SDG03']

--- REASONING ---
SDG01 0.493 → ["abstract: ['inequit*', 'inequ*', 'income', 'distribution*', 'hazards', 'insurance', 'Asia']"]
SDG03 0.493 → ["title: ['disease']", "abstract: ['disease', 'diseases', 'health']"]
SDG10 0.122 → ["abstract: ['health inequity']"]


In [7]:
# SDGS ASLI: SDG 3| SDG 4| SDG 5| SDG 8| SDG 10

title = "Global fertility in 204 countries and territories, 1950–2021,with forecasts to 2100: a comprehensive demographicanalysis for the Global Burden of Disease Study 2021"
abstract = """
Background Accurate assessments of current and future fertility—including overall trends and changing populationage structures across countries and regions—are essential to help plan for the profound social, economic,environmental, and geopolitical challenges that these changes will bring. Estimates and projections of fertility arenecessary to inform policies involving resource and health-care needs, labour supply, education, gender equality, andfamily planning and support. The Global Burden of Diseases, Injuries, and Risk Factors Study (GBD) 2021 producedup-to-date and comprehensive demographic assessments of key fertility indicators at global, regional, and nationallevels from 1950 to 2021 and forecast fertility metrics to 2100 based on a reference scenario and key policy-dependentalternative scenarios.
Methods To estimate fertility indicators from 1950 to 2021, mixed-effects regression models and spatiotemporalGaussian process regression were used to synthesise data from 8709 country-years of vital and sample registrations,1455 surveys and censuses, and 150 other sources, and to generate age-specific fertility rates (ASFRs) for 5-year agegroups from age 10 years to 54 years. ASFRs were summed across age groups to produce estimates of total fertilityrate (TFR). Livebirths were calculated by multiplying ASFR and age-specific female population, then summing acrossages 10–54 years. To forecast future fertility up to 2100, our Institute for Health Metrics and Evaluation (IHME)forecasting model was based on projections of completed cohort fertility at age 50 years (CCF50; the average numberof children born over time to females from a specified birth cohort), which yields more stable and accurate measuresof fertility than directly modelling TFR. CCF50 was modelled using an ensemble approach in which three sub-models(with two, three, and four covariates variously consisting of female educational attainment, contraceptive met need,population density in habitable areas, and under-5 mortality) were given equal weights, and analyses were conductedutilising the MR-BRT (meta-regression—Bayesian, regularised, trimmed) tool. To capture time-series trends inCCF50 not explained by these covariates, we used a first-order autoregressive model on the residual term. CCF50 as aproportion of each 5-year ASFR was predicted using a linear mixed-effects model with fixed-effects covariates (femaleeducational attainment and contraceptive met need) and random intercepts for geographical regions. Projected TFRswere then computed for each calendar year as the sum of single-year ASFRs across age groups. The reference forecastis our estimate of the most likely fertility future given the model, past fertility, forecasts of covariates, and historicalrelationships between covariates and fertility. We additionally produced forecasts for multiple alternative scenarios ineach location: the UN Sustainable Development Goal (SDG) for education is achieved by 2030; the contraceptive metneed SDG is achieved by 2030; pro-natal policies are enacted to create supportive environments for those who givebirth; and the previous three scenarios combined. Uncertainty from past data inputs and model estimation waspropagated throughout analyses by taking 1000 draws for past and present fertility estimates and 500 draws for futureforecasts from the estimated distribution for each metric, with 95% uncertainty intervals (UIs) given as the2·5 and 97·5 percentiles of the draws. To evaluate the forecasting performance of our model and others, we computedskill values—a metric assessing gain in forecasting accuracy—by comparing predicted versus observed ASFRs fromthe past 15 years (2007–21). A positive skill metric indicates that the model being evaluated performs better than thebaseline model (here, a simplified model holding 2007 values constant in the future), and a negative metric indicatesthat the evaluated model performs worse than baseline.
Findings During the period from 1950 to 2021, global TFR more than halved, from 4·84 (95% UI 4·63–5·06) to2·23 (2·09–2·38). Global annual livebirths peaked in 2016 at 142 million (95% UI 137–147), declining to129 million (121–138) in 2021. Fertility rates declined in all countries and territories since 1950, with TFR remainingabove 2·1—canonically considered replacement-level fertility—in 94 (46·1%) countries and territories in 2021. Thisincluded 44 of 46 countries in sub-Saharan Africa, which was the super-region with the largest share of livebirthsin 2021 (29·2% [28·7–29·6]). 47 countries and territories in which lowest estimated fertility between 1950 and 2021was below replacement experienced one or more subsequent years with higher fertility; only three of these locationsrebounded above replacement levels. Future fertility rates were projected to continue to decline worldwide, reachinga global TFR of 1·83 (1·59–2·08) in 2050 and 1·59 (1·25–1·96) in 2100 under the reference scenario. The number ofcountries and territories with fertility rates remaining above replacement was forecast to be 49 (24·0%) in 2050 andonly six (2·9%) in 2100, with three of these six countries included in the 2021 World Bank-defined low-income group,all located in the GBD super-region of sub-Saharan Africa. The proportion of livebirths occurring in sub-Saharan Africawas forecast to increase to more than half of the world’s livebirths in 2100, to 41·3% (39·6–43·1) in 2050 and54·3% (47·1–59·5) in 2100. The share of livebirths was projected to decline between 2021 and 2100 in most of thesix other super-regions—decreasing, for example, in south Asia from 24·8% (23·7–25·8) in 2021 to 16·7% (14·3–19·1)in 2050 and 7·1% (4·4–10·1) in 2100—but was forecast to increase modestly in the north Africa and Middle East andhigh-income super-regions. Forecast estimates for the alternative combined scenario suggest that meeting SDGtargets for education and contraceptive met need, as well as implementing pro-natal policies, would result in globalTFRs of 1·65 (1·40–1·92) in 2050 and 1·62 (1·35–1·95) in 2100. The forecasting skill metric values for the IHMEmodel were positive across all age groups, indicating that the model is better than the constant prediction.
Interpretation Fertility is declining globally, with rates in more than half of all countries and territories in 2021 belowreplacement level. Trends since 2000 show considerable heterogeneity in the steepness of declines, and only a smallnumber of countries experienced even a slight fertility rebound after their lowest observed rate, with none reachingreplacement level. Additionally, the distribution of livebirths across the globe is shifting, with a greater proportionoccurring in the lowest-income countries. Future fertility rates will continue to decline worldwide and will remain loweven under successful implementation of pro-natal policies. These changes will have far-reaching economic andsocietal consequences due to ageing populations and declining workforces in higher-income countries, combinedwith an increasing share of livebirths among the already poorest regions of the world
"""
keywords = ""

result = classify_text_rule_based(title, abstract, keywords)

print("TOP SDG:", result["top_sdg"])
print("DETECTED SDGs:", result["labels"])

print("\n--- REASONING ---")
for sdg, d in result["details"].items():
    if d["confidence"] > 0:
        print(sdg, d["confidence"], "→", d["reasoning"])


TOP SDG: SDG03
DETECTED SDGs: ['SDG01', 'SDG03', 'SDG04', 'SDG05', 'SDG08', 'SDG16']

--- REASONING ---
SDG01 0.4 → ["abstract: ['world bank', 'policy', 'policies', 'low-income', 'gender', 'income', 'distribution*', 'economic', 'Africa', 'Asia', 'equality', 'poorest', 'polic*']", 'soft-exclude applied']
SDG03 0.581 → ["title: ['disease']", "abstract: ['diseases', 'health', 'mortality', 'contraceptive', 'Africa', 'under-5 mortality']"]
SDG04 0.526 → ["abstract: ['education', 'better', 'tool', 'gender', 'child*', 'increase', 'increasing', 'performance', 'children', 'education*', 'skill', 'skill*', 'development', 'Skill', 'develop*', 'Develop*', 'Africa*', 'Africa', 'sustainable', 'sustainable development', 'educat*', 'sustainab*', 'Gender', 'sustainable* develop*', 'environment*', 'social', 'environmental', 'Sustainable', 'policy', 'policies', 'Sustainab*', 'equality', 'needs']"]
SDG05 0.526 → ["abstract: ['gender* equ*', 'gender', 'Gender', 'equal*', 'child*', 'education', 'equality', '